# Alisha Bubbzieee — QLoRA Fine-Tuning on Kaggle (30h Free T4)
### Fine-tuning Qwen/Qwen3.5-2B on 1,389 authentic conversation pairs

**Why Kaggle:**
- 30 free GPU hours per week (Dual T4)
- 12-hour continuous execution
- Checkpoints persist directly into Kaggle Dataset
- Outputs GGUF quantized to Q4_K_M (~1.4 GB) for Redmi Note 11 deployment

In [ ]:
# 1. Install Unsloth and modern training dependencies
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install datasets transformers
!pip install llama-cpp-python

In [ ]:
# 2. Load Base Model with 4-bit Quantization
from unsloth import FastLanguageModel
import torch

max_seq_length = 1024
model_name = "Qwen/Qwen3.5-2B"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    load_in_4bit=True,
    dtype=torch.float16,
)

In [ ]:
# 3. Configure LoRA Parameters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

In [ ]:
# 4. Format Dataset with Alisha Persona Template
from datasets import load_dataset

# Upload chat_train.jsonl and chat_val.jsonl to your Kaggle Notebook
dataset = load_dataset("json", data_files={"train": "chat_train.jsonl", "val": "chat_val.jsonl"})

prompt_template = """<|im_start|>system
{system}<|im_end|>
<|im_start|>user
{human}<|im_end|>
<|im_start|>assistant
{gpt}<|im_end|>"""

def formatting_prompts_func(examples):
    texts = []
    for convs in examples["conversations"]:
        sys = convs[0]["value"]
        hum = convs[1]["value"]
        gpt = convs[2]["value"]
        text = prompt_template.format(system=sys, human=hum, gpt=gpt)
        texts.append(text)
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched=True)

In [ ]:
# 5. Execute SFT Training
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset["train"],
    eval_dataset=dataset["val"],
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        warmup_ratio=0.05,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=42,
        output_dir="alisha_checkpoints",
        save_strategy="steps",
        save_steps=50,
        eval_strategy="steps",
        eval_steps=50,
    ),
)

trainer_stats = trainer.train()

In [ ]:
# 6. Export Merged GGUF for Redmi Note 11 (Q4_K_M Quantization)
# Produces ~1.4GB model ready to run in llama.cpp on device
model.save_pretrained_gguf(
    "alisha_bubbzieee_2b_q4km",
    tokenizer,
    quantization_method="q4_k_m",
)
print("Exported alisha_bubbzieee_2b_q4km.gguf successfully!")

In [ ]:
# 7. Quick Validation Inference
FastLanguageModel.for_inference(model)
inputs = tokenizer(
    [prompt_template.format(
        system="You are Alisha (@iishazsr). Text naturally in lowercase, casual slang, short fragments.",
        human="Friend: did you finish that thing or did you forget again",
        gpt=""
    )],
    return_tensors="pt"
).to("cuda")

outputs = model.generate(**inputs, max_new_tokens=64, use_cache=True, temperature=0.7)
print(tokenizer.batch_decode(outputs)[0])